# 03. Verifying a run with `rustamps verify`

`rustamps verify --run RUN --golden GOLDEN` compares every production artifact. This notebook exercises it on four
cases and checks each outcome against an independent h5py comparison, so we also learn whether `verify` itself can be trusted.

1. golden against itself: must pass;
2. the real Rustamps run against golden: report exactly what it flags;
3. a golden copy with one value changed: must fail on that variable;
4. a golden copy with a file removed: must fail as a missing artifact.

> **Scope.** These notebooks prove Stages 1-5 on `PATCH_1` of the golden dataset, starting from the raw
> `pscands.1.*` candidate files and ending at the Stage-5 patch products. Stages 6-8 are **not covered**:
> they need the merged four-patch golden data, which is not part of the local download.

In [ ]:
import sys
from pathlib import Path

here = Path.cwd().resolve()
root = next(p for p in (here, *here.parents) if (p / "Cargo.toml").exists())
sys.path[:0] = [str(root), str(root / "notebooks")]

import matplotlib.pyplot as plt, numpy as np, pandas as pd
from IPython.display import display, Markdown
import rustamps_nb as nb
pd.set_option("display.width", 200); pd.set_option("display.max_colwidth", 110); pd.set_option("display.max_rows", 120)
print("binary:", nb.binary()); print("golden:", nb.require_golden())

In [ ]:
import json, os, shutil, subprocess
run, manifest = nb.run_chain()
W = nb.WORK
gold = W / "gold1"; shutil.rmtree(gold, ignore_errors=True); (gold / "PATCH_1").mkdir(parents=True)
for f in (nb.GOLDEN / "PATCH_1").iterdir():
    (gold / "PATCH_1" / f.name).symlink_to(f)
(gold / "patch.list").write_text("PATCH_1\n"); (gold / "parms.mat").symlink_to(nb.GOLDEN / "parms.mat")

def verify(run_dir, profile="strict"):
    rc, out, err, _ = nb.cli("verify", "--run", run_dir, "--golden", gold, "--through-stage", 4, "--profile", profile, check=False)
    return rc, json.loads(out[out.find("{"):])
proof = []

## 1. Golden against itself

In [ ]:
rc, rep = verify(gold)
proof.append(nb.check(rc == 0 and rep["ok"] and rep["checked"] == 9 and not rep["failed"], "golden vs golden passes", f"checked={rep['checked']}"))

## 2. The real run against golden

`verify` reports the **first** differing variable of each file. We require that every file it flags is also within the
declared bound according to the independent comparison; this documents precisely how far the Rust run is from MATLAB.

In [ ]:
rc, rep = verify(run, "scientific")
flagged = pd.DataFrame([{"file": f["path"].split("/")[-1], "variable": f["failing_key"], "max_abs": f["max_abs"], "message": f["message"][:70]} for f in rep["failed"]])
display(flagged)
independent = nb.compare_files(run, ["pm1.mat", "select1.mat", "weed1.mat"])
bound_ok = independent.set_index(["file", "variable"]).loc[list(zip(flagged.file, flagged.variable)), "pass"] if len(flagged) else pd.Series([True])
proof.append(nb.check(rc == (0 if rep["ok"] else 1) and rep["checked"] == 9, "verify exit status agrees with its own report", f"rc={rc}, ok={rep['ok']}"))
proof.append(nb.check(bool(bound_ok.all()), f"every file verify flags ({len(flagged)} of {rep['checked']}) is within its declared float bound",
                      "; ".join(f"{a}:{b}={c:.2e}" for a, b, c in zip(flagged.file, flagged.variable, flagged.max_abs))))

## 3. One changed value must be caught

In [ ]:
bad = W / "bad_value"; shutil.rmtree(bad, ignore_errors=True); shutil.copytree(gold, bad, symlinks=True)
target = bad / "PATCH_1" / "select1.mat"; target.unlink(); nb.perturb_copy(nb.GOLDEN / "PATCH_1" / "select1.mat", target, "coh_ps2", 1e-3)
rc, rep = verify(bad)
hit = [f for f in rep["failed"] if f["path"].endswith("select1.mat")]
proof.append(nb.check(rc != 0 and not rep["ok"] and hit and hit[0]["failing_key"] == "coh_ps2",
                      "verify fails on a one-element change", f"{hit[0]['failing_key']}: {hit[0]['message'][:60]}"))

## 4. A missing file must be caught

In [ ]:
miss = W / "missing_file"; shutil.rmtree(miss, ignore_errors=True); shutil.copytree(gold, miss, symlinks=True)
(miss / "PATCH_1" / "weed1.mat").unlink()
rc, rep = verify(miss)
hit = [f for f in rep["failed"] if f["path"].endswith("weed1.mat")]
proof.append(nb.check(rc != 0 and hit and "missing" in hit[0]["message"], "verify fails on a missing artifact", hit[0]["message"]))
for d in (bad, miss): shutil.rmtree(d)

## Verdict

In [ ]:
verdict = pd.DataFrame(proof); display(verdict[["claim", "ok", "evidence"]]); assert verdict.ok.all(); print("NOTEBOOK PASSED:", len(verdict), "checks")